# 00 · What Is OpenUSD?

> Companion notebook to `docs/what-openusd/` from the Learn OpenUSD curriculum.

## Learning objectives
- Understand what OpenUSD (Universal Scene Description) is and where it came from.
- See why industries beyond film (digital twins, simulation, AI for 3D worlds) are adopting it.
- Grasp OpenUSD's core purpose: managing complex 3D scenes across collaborating teams.
- Identify three key capabilities: non-destructive collaboration, modularity/scalability, and cross-application interoperability.
- Confirm your local `usd-core` install works and create your very first `Usd.Stage`.

## How to use this notebook
Run cells top-to-bottom. Generated files land in `./_assets/` next to this notebook. Safe to delete that folder any time.

In [1]:
import sys
print("Python:", sys.version.split()[0])

from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK — usd-core is installed.")

from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK — usd-core is installed.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/_assets


In [2]:
# Try the repo's helper; fall back to an inline version if running outside the uv env.
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded — 3D previews available via DisplayUSD().")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers (lousd not installed).")
    _HAVE_LOUSD = False

lousd helpers loaded — 3D previews available via DisplayUSD().


## 1. What is OpenUSD?

[OpenUSD](https://docs.nvidia.com/learn-openusd/latest/glossary.html) (Universal Scene Description) is an open-source framework originally built at Pixar for describing, composing, simulating, and collaborating on 3D scenes. It began life inside Pixar's animation pipeline but has since spread into architecture, manufacturing, robotics, and AI — anywhere teams need a shared, scalable way to talk about 3D data. Think of it less as "a file format" and more as a language plus runtime for 3D worlds.

In [3]:
# The pxr package is the Python binding for OpenUSD's C++ libraries.
# Let's peek at a few of the modules we'll use throughout this course.
import pxr

modules_of_interest = [
    "Usd",       # Core stage / prim / attribute API
    "Sdf",       # Scene Description Foundations: layers, paths, specs
    "UsdGeom",   # Geometry schemas (Xform, Mesh, Sphere, ...)
    "UsdShade",  # Materials and shading networks
    "Gf",        # Graphics Foundations: vectors, matrices, colors
]
for name in modules_of_interest:
    mod = getattr(pxr, name, None)
    status = "ok" if mod is not None else "MISSING"
    print(f"pxr.{name:<8} {status}")

# Print the USD library version so future-you can reproduce results.
print("\nUSD library version:", Usd.GetVersion())

pxr.Usd      ok
pxr.Sdf      ok
pxr.UsdGeom  ok
pxr.UsdShade ok
pxr.Gf       ok

USD library version: (0, 25, 11)


**What just happened:** we imported the `pxr` Python package and confirmed that the five modules we rely on most are available. `Usd.GetVersion()` returns a `(major, minor, patch)` tuple — handy to record in bug reports.

## 2. Why industries are adopting OpenUSD

OpenUSD caught on in film first because Pixar needed dozens of departments (modeling, rigging, layout, animation, lighting, FX) to touch the same shot without stepping on each other. Today the same problem shows up in factories (digital twins), autonomous-vehicle simulation, and generative AI for 3D: many tools, many teams, one virtual world. OpenUSD provides the shared substrate so none of those groups has to agree on a single DCC. See the [glossary entry on Asset](https://docs.nvidia.com/learn-openusd/latest/glossary.html) for how reusable units flow through these pipelines.

## 3. Core purpose: composing complex scenes

OpenUSD's central job is to let you describe a scene as a composition of smaller pieces instead of one giant blob. A [Stage](https://docs.nvidia.com/learn-openusd/latest/glossary.html) is the runtime view of that composed scene, and [Layers](https://docs.nvidia.com/learn-openusd/latest/glossary.html) are the files that contribute data to it. The upcoming notebooks dig into this in depth — for now, let's just build the smallest possible stage and look at it.

In [4]:
# Create the very first stage of the course. A stage is backed by a root layer
# on disk; we put it under _assets/ so it's easy to clean up.
stage = create_new_stage("_assets/00_hello_usd.usda")

# A brand-new stage has no user prims yet, but it already has a pseudo-root
# (represented by the path "/") that every other prim hangs off of.
print("Stage root layer:", stage.GetRootLayer().identifier)
print("Pseudo-root prim:", stage.GetPseudoRoot().GetPath())
print("Number of user prims:", sum(1 for _ in stage.Traverse()))

# Save the (empty) stage so the .usda file exists on disk.
stage.GetRootLayer().Save()
print("Saved empty stage.")

Stage root layer: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/_assets/00_hello_usd.usda
Pseudo-root prim: /
Number of user prims: 0
Saved empty stage.


**What just happened:** `create_new_stage` wrote a fresh `.usda` (ASCII USD) file under `_assets/`. The stage is empty apart from the implicit pseudo-root at path `/`. `stage.Traverse()` walks every prim below the pseudo-root — right now that's zero.

## 4. Key capability #1: non-destructive collaboration

In OpenUSD, edits are stored as *layers* that compose together rather than as in-place mutations of a single file. Two artists can touch the same scene by each owning their own layer; a [LayerStack](https://docs.nvidia.com/learn-openusd/latest/glossary.html) merges their contributions with clear precedence rules. The original data is preserved, and any layer can be muted or swapped without losing work. You will build real sublayers in notebook 02; for now we just create a prim so the file has something visible.

In [5]:
# Define a single Xform prim to demonstrate that the stage can carry content.
# UsdGeom.Xform is the standard transformable container prim in OpenUSD.
root_xform = UsdGeom.Xform.Define(stage, "/World")
print("Defined prim:", root_xform.GetPath(), "type=", root_xform.GetPrim().GetTypeName())

# Persist the edit to the root layer.
stage.GetRootLayer().Save()

# Read the file back as text — .usda is human-readable by design, which is
# exactly what makes OpenUSD diff- and merge-friendly.
usda_text = Path("_assets/00_hello_usd.usda").read_text()
print("\n--- _assets/00_hello_usd.usda ---")
print(usda_text)

Defined prim: /World type= Xform

--- _assets/00_hello_usd.usda ---
#usda 1.0

def Xform "World"
{
}




**What just happened:** we added `/World` (an `Xform`) to the stage and saved it. Notice the serialized `.usda` is plain text — that's what lets version control, code review, and automation treat USD edits the same way we treat source code.

## 5. Key capability #2: modularity and scalability

Scenes in OpenUSD are built out of reusable pieces. A [Reference](https://docs.nvidia.com/learn-openusd/latest/glossary.html) pulls another asset into the current scene; [Instancing](https://docs.nvidia.com/learn-openusd/latest/glossary.html) lets a single prototype back thousands of copies cheaply. That's how you scale from one prop to an entire virtual city without duplicating data. We will explore references and payloads later — here we just prove the stage can hold a named sub-tree that could later become an asset.

In [6]:
# Add a child prim under /World to sketch what a reusable "props" module
# might look like. Each of these paths could eventually come from its own
# referenced asset layer.
UsdGeom.Xform.Define(stage, "/World/Props")
UsdGeom.Xform.Define(stage, "/World/Props/Chair")
UsdGeom.Xform.Define(stage, "/World/Props/Table")

# Traverse the stage so we can see the hierarchy we just built.
print("Prims on stage:")
for prim in stage.Traverse():
    print(" ", prim.GetPath(), "(", prim.GetTypeName() or "<untyped>", ")")

stage.GetRootLayer().Save()

Prims on stage:
  /World ( Xform )
  /World/Props ( Xform )
  /World/Props/Chair ( Xform )
  /World/Props/Table ( Xform )


True

**What just happened:** we grew the namespace into a tiny hierarchy. In a real production, `/World/Props/Chair` would typically be a reference to a standalone chair asset layer — so swapping the chair asset updates every scene that uses it.

## 6. Key capability #3: cross-application interoperability

OpenUSD acts as a neutral interchange between DCCs (digital content creation tools) like Maya, Houdini, Blender, Unreal, and Omniverse. Because schemas such as `UsdGeom` and `UsdShade` are standardized, a mesh authored in one tool means the same thing in another. The textual `.usda` you just saw is one of three equivalent encodings — the others are the binary `.usdc` (crate) format and the `.usdz` zipped package — and OpenUSD will pick between them transparently.

In [7]:
# Export the same stage to the binary crate format. Every supported DCC
# can read either file; the choice is a storage/perf tradeoff, not a
# semantic one.
crate_path = NB_DIR / "_assets" / "00_hello_usd.usdc"
stage.GetRootLayer().Export(str(crate_path))

usda_size = (NB_DIR / "_assets" / "00_hello_usd.usda").stat().st_size
usdc_size = crate_path.stat().st_size
print(f"usda (ascii ) : {usda_size:>5} bytes")
print(f"usdc (binary) : {usdc_size:>5} bytes")

# Re-open the crate file to prove it round-trips through OpenUSD cleanly.
reopened = Usd.Stage.Open(str(crate_path))
print("\nPrims in the reopened crate file:")
for prim in reopened.Traverse():
    print(" ", prim.GetPath())

usda (ascii ) :   161 bytes
usdc (binary) :   670 bytes

Prims in the reopened crate file:
  /World
  /World/Props
  /World/Props/Chair
  /World/Props/Table


**What just happened:** we exported the ASCII stage to a binary `.usdc` crate file and reopened it. The prim hierarchy survives the round trip unchanged — that's the essence of OpenUSD's interoperability guarantee.

## Key takeaways

- **OpenUSD is a framework, not just a file format.** It provides APIs, schemas, a composition engine, and multiple on-disk encodings (`.usda`, `.usdc`, `.usdz`).
- **It was built to solve collaboration at scale.** Multiple teams, multiple tools, one shared virtual scene — without destructive overwrites.
- **Three pillars matter most early on:** non-destructive layered edits, modular/reusable assets, and cross-DCC interoperability.
- **A stage is the live, composed view of a scene,** and every prim lives under a pseudo-root at path `/`.
- **Text-based `.usda` is a superpower for tooling** — you can diff, review, and script USD changes the same way you do source code.

See the [Learn OpenUSD glossary](https://docs.nvidia.com/learn-openusd/latest/glossary.html) any time a term feels fuzzy.

## Next up

Continue with **`01_stage_setting.ipynb`**, where we slow down and focus on the stage itself — how to create one, inspect it, populate it with typed prims, and save it out.